In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

path  = "C:\\Users\\User0\\PycharmProjects\\zoomcamp-hw\\homeworks\\03-classification\\data\\course_lead_scoring_2026.csv"
df = pd.read_csv(path)

In [70]:
df.shape


(5000, 9)

In [71]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [72]:
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [74]:
df.isna().sum().sort_values(ascending=False)

annual_income               369
industry                    240
location                    208
employment_status           194
lead_source                 148
lead_score                   35
number_of_courses_viewed      0
interaction_count             0
converted                     0
dtype: int64

In [75]:
target = 'converted'

categorical_features = df.select_dtypes(include=['object', 'str']).columns.tolist()

numerical_features = [
    col for col in df.select_dtypes(include=['int64', 'float64']).columns
    if col != target
]

In [76]:
categorical_features_missing = df[categorical_features].isna().sum().sort_values(ascending=False)
print("Categorical features with missing values:\n", categorical_features_missing[categorical_features_missing > 0])

numerical_features_missing = df[numerical_features].isna().sum().sort_values(ascending=False)
print("Numerical features with missing values:\n", numerical_features_missing[numerical_features_missing > 0])

Categorical features with missing values:
 industry             240
location             208
employment_status    194
lead_source          148
dtype: int64
Numerical features with missing values:
 annual_income    369
lead_score        35
dtype: int64


For the question 1, lets fill categorical missing features with "NA", and numerical missing features with 0


In [77]:
df[categorical_features] = df[categorical_features].fillna("NA")
df[numerical_features] = df[numerical_features].fillna(0)

In [78]:
df.isna().sum().sort_values(ascending=False)

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [79]:
industry_mode = df["industry"].mode()[0]
print("Most frequent industry:", industry_mode)

Most frequent industry: technology


In [80]:
print(df[numerical_features].corr())

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  


In [81]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [82]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [83]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [84]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [85]:
from sklearn.metrics import mutual_info_score
for f in ["industry","location","lead_source","employment_status"]:
    print(f"Mutual information score for {f}: {round((mutual_info_score(df_train[f], y_train)),2)}")

Mutual information score for industry: 0.0
Mutual information score for location: 0.0
Mutual information score for lead_source: 0.03
Mutual information score for employment_status: 0.02


question 4: logistic regression

In [87]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)
train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print("Accuracy on validation set:", accuracy)

Accuracy on validation set: 0.645


In [91]:
baseline_accuracy = accuracy

features_to_understand = ["lead_source", "number_of_courses_viewed", "interaction_count"]

for feature in features_to_understand:
    features = [f for f in df_train.columns if f != feature]
    dv = DictVectorizer(sparse=False)
    train_dicts = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    val_dicts = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy_specific = accuracy_score(y_val, y_pred)
    print(f"Accuracy on validation set without {feature}: {accuracy_specific}")

Accuracy on validation set without lead_source: 0.642
Accuracy on validation set without number_of_courses_viewed: 0.643
Accuracy on validation set without interaction_count: 0.601


q6

In [92]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)
train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    print(f"Accuracy on validation set with C={C}: {accuracy}")

Accuracy on validation set with C=1e-06: 0.598
Accuracy on validation set with C=1e-05: 0.598
Accuracy on validation set with C=0.0001: 0.613
Accuracy on validation set with C=0.001: 0.645
